In [1]:
import pandas as pd

tx = pd.read_csv("/content/transaction_data.csv")
prod = pd.read_csv("/content/product.csv")
hh = pd.read_csv("/content/hh_demographic.csv")

for name, df in [("transactions", tx), ("products", prod), ("households", hh)]:
    print("=====", name, df.shape)
    print(df.dtypes)
    print("missing values:\n", df.isna().sum())
    print("duplicate rows:", df.duplicated().sum())

===== transactions (2595732, 12)
household_key          int64
BASKET_ID              int64
DAY                    int64
PRODUCT_ID             int64
QUANTITY               int64
SALES_VALUE          float64
STORE_ID               int64
RETAIL_DISC          float64
TRANS_TIME             int64
WEEK_NO                int64
COUPON_DISC          float64
COUPON_MATCH_DISC    float64
dtype: object
missing values:
 household_key        0
BASKET_ID            0
DAY                  0
PRODUCT_ID           0
QUANTITY             0
SALES_VALUE          0
STORE_ID             0
RETAIL_DISC          0
TRANS_TIME           0
WEEK_NO              0
COUPON_DISC          0
COUPON_MATCH_DISC    0
dtype: int64
duplicate rows: 0
===== products (92353, 7)
PRODUCT_ID               int64
MANUFACTURER             int64
DEPARTMENT              object
BRAND                   object
COMMODITY_DESC          object
SUB_COMMODITY_DESC      object
CURR_SIZE_OF_PRODUCT    object
dtype: object
missing values:
 PRODUCT

In [2]:
tx.describe()

,household_key,BASKET_ID,DAY,PRODUCT_ID,QUANTITY,SALES_VALUE,STORE_ID,RETAIL_DISC,TRANS_TIME,WEEK_NO,COUPON_DISC,COUPON_MATCH_DISC
count,2.595732e+06,2.595732e+06,2.595732e+06,2.595732e+06,2.595732e+06,2.595732e+06,2.595732e+06,2.595732e+06,2.595732e+06,2.595732e+06,2.595732e+06,2.595732e+06
mean,1.271953e+03,3.402620e+10,3.887562e+02,2.891435e+06,1.004286e+02,3.104120e+00,3.142673e+03,-5.387054e-01,1.561586e+03,5.622150e+01,-1.641600e-02,-2.918564e-03
std,7.260660e+02,4.711649e+09,1.897210e+02,3.837404e+06,1.153436e+03,4.182274e+00,8.937113e+03,1.249191e+00,3.998378e+02,2.710223e+01,2.168410e-01,3.969004e-02
min,1.000000e+00,2.698485e+10,1.000000e+00,2.567100e+04,0.000000e+00,0.000000e+00,1.000000e+00,-1.800000e+02,0.000000e+00,1.000000e+00,-5.593000e+01,-7.700000e+00
25%,6.560000e+02,3.040805e+10,2.290000e+02,9.174590e+05,1.000000e+00,1.290000e+00,3.300000e+02,-6.900000e-01,1.308000e+03,3.300000e+01,0.000000e+00,0.000000e+00
50%,1.272000e+03,3.276081e+10,3.900000e+02,1.028816e+06,1.000000e+00,2.000000e+00,3.720000e+02,-1.000000e-02,1.613000e+03,5.600000e+01,0.000000e+00,0.000000e+00
75%,1.913000e+03,4.012685e+10,5.530000e+02,1.133018e+06,1.000000e+00,3.490000e+00,4.220000e+02,0.000000e+00,1.843000e+03,8.000000e+01,0.000000e+00,0.000000e+00
max,2.500000e+03,4.230536e+10,7.110000e+02,1.831630e+07,8.963800e+04,8.400000e+02,3.428000e+04,3.990000e+00,2.359000e+03,1.020000e+02,0.000000e+00,0.000000e+00


In [3]:
# make sure product table is loaded
prod = pd.read_csv("/content/product.csv")

rows_start = len(tx)

# 1. remove duplicates
tx = tx.drop_duplicates()

# 2. remove rows with zero quantity or zero/negative sales
tx = tx[(tx["QUANTITY"] > 0) & (tx["SALES_VALUE"] > 0)]

# 3. fix data types
tx["WEEK_NO"] = tx["WEEK_NO"].astype(int)
tx["PRODUCT_ID"] = tx["PRODUCT_ID"].astype(int)

# 4. add department from the product table
tx = tx.merge(prod[["PRODUCT_ID", "DEPARTMENT"]], on="PRODUCT_ID", how="left")
tx["DEPARTMENT"] = tx["DEPARTMENT"].fillna("UNKNOWN").str.strip()

# 5. create discount fields
tx["TOTAL_DISC"] = -(tx["RETAIL_DISC"] + tx["COUPON_MATCH_DISC"] + tx["COUPON_DISC"])
tx["GROSS_SALES"] = tx["SALES_VALUE"] + tx["TOTAL_DISC"]

print("rows before:", rows_start, "| rows after:", len(tx))
print(tx["DEPARTMENT"].value_counts().head(10))
print("total discounts:", round(tx["TOTAL_DISC"].sum()), "| gross sales:", round(tx["GROSS_SALES"].sum()))
print("overall discount rate: %.1f%%" % (tx["TOTAL_DISC"].sum() / tx["GROSS_SALES"].sum() * 100))

rows before: 2595732 | rows after: 2576815
DEPARTMENT
GROCERY          1642306
DRUG GM           275229
PRODUCE           255604
MEAT-PCKGD        110794
MEAT               88080
DELI               62554
PASTRY             37470
NUTRITION          32084
KIOSK-GAS          22059
SEAFOOD-PCKGD      11074
Name: count, dtype: int64
total discounts: 1423420 | gross sales: 9480863
overall discount rate: 15.0%


In [4]:
import plotly.express as px

# Chart 1: discount rate over time
weekly = tx.groupby("WEEK_NO")[["GROSS_SALES","TOTAL_DISC","SALES_VALUE"]].sum().reset_index()
weekly["DISC_PCT"] = weekly["TOTAL_DISC"] / weekly["GROSS_SALES"] * 100
px.line(weekly, x="WEEK_NO", y="DISC_PCT",
        title="Weekly Discount Rate (% of Gross Sales)",
        labels={"WEEK_NO":"Week","DISC_PCT":"Discount Rate (%)"}).show()

# Department summary (keep only departments with meaningful sales)
dept = tx.groupby("DEPARTMENT")[["GROSS_SALES","TOTAL_DISC","SALES_VALUE","QUANTITY"]].sum().reset_index()
dept["DISC_PCT"] = dept["TOTAL_DISC"] / dept["GROSS_SALES"] * 100
dept = dept[dept["GROSS_SALES"] > 5000].sort_values("DISC_PCT")

# Chart 2: discount rate by department
px.bar(dept, x="DISC_PCT", y="DEPARTMENT", orientation="h",
       title="Discount Rate by Department (%)",
       labels={"DISC_PCT":"Discount Rate (%)","DEPARTMENT":"Department"}).show()

# Chart 3: discount depth vs revenue
px.scatter(dept, x="DISC_PCT", y="SALES_VALUE", size="QUANTITY", hover_name="DEPARTMENT",
           title="Discount Rate vs Net Revenue by Department",
           labels={"DISC_PCT":"Discount Rate (%)","SALES_VALUE":"Net Revenue ($)"}).show()

# Chart 4: correlation heatmap
px.imshow(tx[["QUANTITY","SALES_VALUE","TOTAL_DISC","GROSS_SALES"]].corr(),
          text_auto=".2f", title="Correlation Between Key Variables").show()

# Print the department table so you can read the numbers
print(dept.sort_values("DISC_PCT", ascending=False).round(1).to_string(index=False))

     DEPARTMENT  GROSS_SALES  TOTAL_DISC  SALES_VALUE  QUANTITY  DISC_PCT
  SEAFOOD-PCKGD      84101.0     21007.4      63093.6     13896      25.0
           MEAT     724665.2    175878.7     548786.5    118940      24.3
     MEAT-PCKGD     529811.2    117374.4     412436.8    147194      22.2
        SEAFOOD      35957.3      6460.4      29496.9      4553      18.0
      COSMETICS      39294.2      6933.8      32360.4      7919      17.6
        GROCERY    4936969.2    843158.4    4093810.8   2193549      17.1
         PASTRY     137679.3     15939.4     121739.9     49371      11.6
      NUTRITION     110085.6     12416.6      97669.0     43240      11.3
        PRODUCE     625450.1     68012.2     557437.9    319684      10.9
  GARDEN CENTER       8562.0       894.2       7667.8      1325      10.4
        DRUG GM    1165010.7    109653.6    1055357.2    353162       9.4
           DELI     283301.6     22435.6     260865.9     66957       7.9
      KIOSK-GAS     561447.7     17225

In [5]:
summary = tx.groupby(["WEEK_NO","DEPARTMENT"])[["SALES_VALUE","TOTAL_DISC","GROSS_SALES"]].sum().reset_index()
summary.to_csv("summary.csv", index=False)
open("requirements.txt","w").write("streamlit\npandas\nplotly\n")
print(summary.shape)

(2430, 5)


In [6]:
%%writefile app.py
import streamlit as st, pandas as pd, plotly.express as px

st.set_page_config(page_title="Retail Discount Analytics", layout="wide")
df = pd.read_csv("summary.csv")
st.title("Retail Discount Strategy & Profitability Dashboard")

depts = st.sidebar.multiselect("Department", sorted(df.DEPARTMENT.unique()), default=sorted(df.DEPARTMENT.unique()))
wmin, wmax = int(df.WEEK_NO.min()), int(df.WEEK_NO.max())
weeks = st.sidebar.slider("Week range", wmin, wmax, (wmin, wmax))
d = df[df.DEPARTMENT.isin(depts) & df.WEEK_NO.between(*weeks)]

if d.empty:
    st.warning("No data for this selection.")
    st.stop()

c1, c2, c3 = st.columns(3)
c1.metric("Net Revenue", f"${d.SALES_VALUE.sum():,.0f}")
c2.metric("Total Discounts", f"${d.TOTAL_DISC.sum():,.0f}")
c3.metric("Discount Rate", f"{d.TOTAL_DISC.sum()/d.GROSS_SALES.sum()*100:.1f}%")

wk = d.groupby("WEEK_NO")[["TOTAL_DISC","GROSS_SALES"]].sum().reset_index()
wk["Discount Rate (%)"] = wk.TOTAL_DISC / wk.GROSS_SALES * 100
st.plotly_chart(px.line(wk, x="WEEK_NO", y="Discount Rate (%)", title="Weekly Discount Rate (%)"), use_container_width=True)

dp = d.groupby("DEPARTMENT")[["SALES_VALUE","TOTAL_DISC","GROSS_SALES"]].sum().reset_index()
dp["Discount Rate (%)"] = dp.TOTAL_DISC / dp.GROSS_SALES * 100
st.plotly_chart(px.bar(dp.sort_values("Discount Rate (%)"), x="Discount Rate (%)", y="DEPARTMENT", orientation="h",
                title="Discount Rate by Department (%)"), use_container_width=True)
st.plotly_chart(px.scatter(dp, x="Discount Rate (%)", y="SALES_VALUE", hover_name="DEPARTMENT",
                title="Discount Rate vs Net Revenue"), use_container_width=True)

Writing app.py


In [8]:
from google.colab import files
for f in ["app.py","summary.csv","requirements.txt"]:
    files.download(f)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>